In [1]:
import json
import urllib.request
import time
from pathlib import Path
from IPython.display import display, Markdown

# =========================================================
# CONFIG
# =========================================================
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "mistral"

# ── 9 zones océaniques ──
ZONES = {
    "Océan Pacifique Sud": {
        "description": "Océan Pacifique Sud, Polynésie, Mélanésie, Micronésie, côtes australiennes, îles Galápagos",
        "escales": ["Île de Crespo", "Archipel des Pomotou", "Archipel des nouvelles Hébrides", "La Pouasie", "Cap Wessel", "Mer de Timor"]
    },
    "Océan Indien": {
        "description": "Océan Indien, côtes indiennes, Sri Lanka, île Keeling, golfe du Bengale, archipel des Maldives",
        "escales": ["Île Keeling", "Golfe du Bengale"]
    },
    "Mer Rouge": {
        "description": "Mer Rouge, golfe d'Aden, détroit de Babel-Mandeb, côtes arabiques et africaines orientales",
        "escales": ["Isthme de Suez"]
    },
    "Méditerranée": {
        "description": "Mer Méditerranée, mer Égée, mer Adriatique, détroit de Gibraltar, archipel grec",
        "escales": ["L'Archipel grec", "Méditerranée", "L'Atlantide"]
    },
    "Océan Atlantique Sud": {
        "description": "Océan Atlantique Sud, mer des Sargasses, côtes africaines, côtes sud-américaines, mer des Caraïbes, Antilles",
        "escales": ["La mer des Sargasses", "Contour Afrique Nord", "Cap Horn", "Martinique et Guadeloupe", "Île Lucayes", "Contour Amérique Sud"]
    },
    "Pôle Sud": {
        "description": "Océan Antarctique, mer de Weddell, banquise antarctique, terres australes",
        "escales": ["Pôle Sud", "Cap Horn (retour)"]
    },
    "Océan Atlantique Nord": {
        "description": "Océan Atlantique Nord, banc de Terre-Neuve, Gulf Stream, côtes américaines, côtes brésiliennes",
        "escales": ["Cap Hatteras", "Long Island", "Terre Neuve", "Irlande"]
    },
    "Mers Celtiques et Manche": {
        "description": "Manche, mer Celtique, côtes anglaises, côtes françaises, mer d'Irlande",
        "escales": ["Manche", "Contour Angleterre Sud", "Contour Angleterre Nord"]
    },
    "Mer de Norvège et Maelström": {
        "description": "Mer de Norvège, mer du Nord, archipel des Lofoten, Maelström, côtes scandinaves",
        "escales": ["Maelström"]
    }
}

# =========================================================
# OLLAMA
# =========================================================
def appeler_ollama(prompt, num_predict=2000):
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.2, "num_predict": num_predict}
    }).encode("utf-8")
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST"
    )
    try:
        with urllib.request.urlopen(req, timeout=180) as resp:
            return json.loads(resp.read().decode("utf-8")).get("response", "").strip()
    except Exception as e:
        return f"Erreur Ollama : {e}"

def prompt_decouvertes(nom_zone, description_zone):
    return f"""Tu es un historien des sciences spécialisé dans le XIXe siècle et l'exploration maritime.

ZONE GÉOGRAPHIQUE : {nom_zone}
DESCRIPTION : {description_zone}

- DATES STRICTEMENT entre 1800 et 1900 — aucune date après 1900 interdite

DOMAINES À COUVRIR (un ou deux par domaine) :
- Naturalistes et biologistes ayant étudié cette zone précise
- Expéditions océanographiques réelles dans cette zone
- Espèces maritimes découvertes et décrites pour la première fois dans cette zone
- Avancées en navigation et cartographie maritime de cette zone
- Découvertes géologiques et bathymétriques

RÈGLES STRICTES :
- Uniquement des faits historiques réels et vérifiables du XIXe siècle
- Cite toujours le nom du scientifique ou de l'expédition, la date précise et la découverte
- Varie les scientifiques — ne cite pas plusieurs fois le même
- Ne répète pas des découvertes qui appartiendraient à d'autres zones océaniques
- Réponse en français uniquement
- Ne mentionne pas Jules Verne ni son roman

- FORMAT STRICT sans numéro ni gras :
🔬 [Nom] ([année]) — [titre court]
   [Une phrase de contexte factuel]
🔬 [Nom du scientifique ou expédition] ([année]) — [titre court de la découverte]
   [Une phrase de contexte précis et factuel]
"""

# =========================================================
# GÉNÉRATION DES 9 ZONES
# =========================================================
resultats = {}

for nom_zone, info in ZONES.items():
    print(f"\n{'='*55}")
    print(f"🌊 {nom_zone}")
    print(f"{'='*55}")
    print("🤖 Ollama en cours...")

    reponse = appeler_ollama(prompt_decouvertes(nom_zone, info["description"]))
    resultats[nom_zone] = reponse

    display(Markdown(f"""
---
### 🌊 {nom_zone}
{reponse}
---
"""))
    time.sleep(2)

# Sauvegarde JSON intermédiaire
with open("decouvertes_zones.json", "w", encoding="utf-8") as f:
    json.dump(resultats, f, ensure_ascii=False, indent=2)

print("\n✅ Toutes les zones générées !")
print("📄 Sauvegardé dans decouvertes_zones.json") 



🌊 Océan Pacifique Sud
🤖 Ollama en cours...



---
### 🌊 Océan Pacifique Sud
🔬 Louis Antoine de Bougainville (1768) — Voyage autour du monde
   Le navigateur français explore pour la première fois l'océan Pacifique Sud, y compris les îles Galápagos.

🔬 HMS Beagle (1835-1836) — Expédition océanographique dans le Pacifique Sud
   L'expédition britannique découvre l'archipel des Chagos et étudie la géologie de l'île d'Easter.

🔬 Charles Darwin (1835) — Voyage sur l'HMS Beagle
   Darwin décrit pour la première fois les iguanes marine à fourreaux sur les îles Galápagos.

🔬 HMS Challenger (1872-1876) — Expédition océanographique dans le Pacifique Sud
   L'expédition britannique effectue des relevés bathymétriques et découvre de nombreuses espèces marines inconnues.

🔬 Thomas Henry Huxley (1873) — Voyage sur l'HMS Challenger
   Huxley étudie les coraux et les mollusques trouvés dans le Pacifique Sud.

🔬 Robert Dampier (1874-1876) — Expédition océanographique dans le Pacifique Sud
   L'expédition britannique découvre de nombreuses espèces marines inconnues, notamment des poissons et des crustacés.

🔬 Jules Dumont d'Urville (1832) — Expédition océanographique dans le Pacifique Sud
   L'expédition française découvre l'archipel des Kerguelen et étudie la géologie de l'île de la Possession.

🔬 John MacGillivray (1839-1841) — Expédition océanographique dans le Pacifique Sud
   L'expédition britannique découvre de nombreuses espèces marines inconnues, notamment des poissons et des crustacés.

🔬 David Livingstone (1867) — Expédition océanographique dans le Pacifique Sud
   L'expédition britannique étudie la géologie de l'île Norfolk et découvre de nombreuses espèces marines inconnues.
---



🌊 Océan Indien
🤖 Ollama en cours...



---
### 🌊 Océan Indien
🔬 Alfred Russel Wallace (1854) — Exploration des côtes indiennes
   Wallace effectue une expédition en Malaisie, Sri Lanka et dans l'océan Indien où il étudie la faune locale.

🔬 Expédition Challenger (1872-1876) — Exploration océanographique du golfe du Bengale
   L'expédition Challenger collecte des données sur la géologie, la botanique et la zoologie dans le golfe du Bengale.

🔬 Henri de Lacaze-Duthiers (1873) — Découverte d'une nouvelle espèce de corail à l'île Keeling
   Lacaze-Duthiers découvre une nouvelle espèce de corail, *Pocillopora damicornis*, lors de son voyage à l'île Keeling.

🔬 Albert Günther (1876) — Découverte d'une nouvelle espèce de requin aux Maldives
   Günther découvre une nouvelle espèce de requin, *Carcharhinus albimarginatus*, lors de son voyage aux Maldives.

🔬 Louis Antoine Gautier (1896) — Avancées en cartographie maritime dans l'océan Indien
   Gautier effectue des relevés hydrographiques et cartographiques dans l'océan Indien, contribuant à la connaissance de la géographie marine de la région.
---



🌊 Mer Rouge
🤖 Ollama en cours...


...


🌊 Mers Celtiques et Manche
🤖 Ollama en cours...



---
### 🌊 Mers Celtiques et Manche
🔬 Louis-Claude de Saulces de Freycinet (1819) — Expédition du Surtout au large des côtes françaises
   L'expédition du Surtout, menée par le capitaine de vaisseau Louis-Claude de Saulces de Freycinet, a cartographié et étudié la géologie de la Manche et des mers celtiques.

🔬 HMS Challenger (1872-1876) — Découverte du geyser hydrothermal sous-marin
   L'expédition HMS Challenger, menée par l'amiral George Strong Nares, a découvert le premier geyser hydrothermal sous-marin dans la mer d'Irlande.

🔬 Jean-Victor Audouin (1826) — Découverte du limace de mer *Haliotis tuberculata*
   Le naturaliste français Jean-Victor Audouin a découvert et décrit pour la première fois le limace de mer *Haliotis tuberculata* dans les mers celtiques.

🔬 Isidore Geoffroy Saint-Hilaire (1824) — Découverte du poisson-chèvre *Sebastes marinus*
   Le naturaliste français Isidore Geoffroy Saint-Hilaire a découvert et décrit pour la première fois le poisson-chèvre *Sebastes marinus* dans les mers celtiques.

🔬 Charles Darwin (1837) — Théorie de l'évolution par sélection naturelle
   Durant son voyage sur le HMS Beagle, le naturaliste britannique Charles Darwin a étudié la faune et la flore des côtes anglaises et françaises. Sa théorie de l'évolution par sélection naturelle a été largement influencée par ses observations dans cette zone.

🔬 Alfred Russel Wallace (1854) — Théorie de la sélection naturelle
   Le naturaliste britannique Alfred Russel Wallace a également étudié la faune et la flore des côtes anglaises et françaises, et a développé sa propre théorie de la sélection naturelle indépendamment de Darwin.
---



🌊 Mer de Norvège et Maelström
🤖 Ollama en cours...



---
### 🌊 Mer de Norvège et Maelström
🔬 Fridtjof Nansen (1873) — Expédition dans la Mer de Norvège
   Le navigateur norvégien Fridtjof Nansen a mené une expédition pour étudier la Mer de Norvège et le Maelström. Il a navigué à bord du "Voyageur Arctique" et a atteint les Lofoten.

🔬 Hjalmar Johansen (1874) — Découverte de la baie d'Isfjorden
   Le navigateur norvégien Hjalmar Johansen a découvert la baie d'Isfjorden dans les Lofoten en 1874. Il était membre de l'expédition menée par Fridtjof Nansen.

🔬 Adolf Erik Nordenskiöld (1869) — Expédition océanographique dans la Mer de Norvège
   L'expédition océanographique suédoise menée par Adolf Erik Nordenskiöld a étudié la Mer de Norvège en 1869. Ils ont collecté des données sur les courants marins et les espèces marines dans cette zone.

🔬 Johan Hjort (1894) — Découverte du saumon blanc
   Le biologiste norvégien Johan Hjort a découvert le saumon blanc dans la Mer de Norvège en 1894. Il a également étudié les populations de poissons dans cette zone et leur impact sur l'écosystème marin.

🔬 Carl Weyprecht (1872-1876) — Expédition polaire internationale
   L'expédition polaire internationale menée par Carl Weyprecht a étudié la Mer de Norvège et les côtes scandinaves entre 1872 et 1876. Ils ont collecté des données sur la navigation, la cartographie maritime et la géologie de cette zone.

🔬 Alfred Wegener (1905) — Découverte de la plaque tectonique
   Bien que l'idée de la plaque tectonique n'ait pas été formulée avant 1912, Alfred Wegener a commencé à étudier les mouvements des continents en 1905. Il a également effectué des recherches dans la Mer de Norvège et a découvert des preuves de la collision entre les plaques européenne et nord-américaine.
---



✅ Toutes les zones générées !
📄 Sauvegardé dans decouvertes_zones.json
